# 既存20epoch runの再開用入口（今回の移行では実行していない）

local → GitHub → Colabで固定ソースを受け渡す。Driveのrun・checkpointをそのまま保持する。
無料GPUの利用と再開epochの明示的な許可を得てから使用する。STOP_AFTER_EPOCHの初期値は5で、学習セルは意図的に停止する。署名・環境不一致を迂回しない。


In [ ]:
from pathlib import Path
import subprocess, sys, json, hashlib, importlib.metadata
from google.colab import drive
drive.mount('/content/drive')
REPO=Path('/content/research')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','research/local-first-20261014','https://github.com/hw02l/depth-anything-v2-uav-rice-research.git',str(REPO)],check=True)
DRIVE=Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
RUN_ID='metric_outdoor_full_20261014_s42_518_cap80_gt65'
RUN=DRIVE/'runs'/RUN_ID
SOURCE=REPO/'experiments'/RUN_ID/'source_full_v1'
META=json.loads((RUN/'run_metadata.json').read_text())
STATUS=json.loads((RUN/'run_status.json').read_text())
assert STATUS['status']=='paused_after_epoch' and STATUS['completed_epochs']==5
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for b in iter(lambda:stream.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
for name,expected in META['resume_signature']['code_sha256'].items():
    path=SOURCE/('config/scene_split_plan.json' if name=='scene_split_plan.json' else name)
    assert sha(path)==expected, name
for split in ('train','validation'):
    assert sha(DRIVE/'uav_metric_finetuning/config/verified_plan'/f'{split}.csv')==META[split+'_manifest_sha256']
for package,expected in META['resume_signature']['runtime_versions'].items():
    assert importlib.metadata.version(package)==expected, f'{package} runtime mismatch; do not bypass'
CP=RUN/'checkpoint_epoch_0005'
marker=json.loads((CP/'complete.json').read_text())
for name,expected in marker['file_sha256'].items(): assert sha(CP/name)==expected,name
assert META['arguments']['epochs']==20
print('Canonical code/manifests/runtime and epoch 5 checkpoint verified. No training has started.')


## 既存DriveデータをSSDへ必要な分だけコピー
512組以外の取得やHugging Faceデータ取得は行わない。固定スクリプトは既存サイズ一致ファイルを再コピーしない。

In [ ]:
stage=[sys.executable,str(SOURCE/'stage_selected_data.py'),
       '--plan-dir',str(DRIVE/'uav_metric_finetuning/config/verified_plan'),
       '--source-rgb-root',str(DRIVE/'data/uav3dcrop_rgb'),
       '--source-depth-root',str(DRIVE/'data/uav3dcrop_depth'),
       '--target-rgb-root','/content/uav3dcrop/rgb',
       '--target-depth-root','/content/uav3dcrop/depth',
       '--splits','train','validation','--stage']
subprocess.run(stage,check=True)


## 新しいセッションのVRAM事前計測
無料GPUのみ使用する。CUDAが使えない・制限・事前計測失敗なら以降を実行しない。

In [ ]:
import torch
from datetime import datetime, timezone
assert torch.cuda.is_available(), 'Free GPU unavailable; stop safely.'
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
PREFLIGHT=DRIVE/'experiments'/RUN_ID/'preflight'/f'migration_resume_{stamp}.json'
a=META['arguments']
subprocess.run([sys.executable,str(SOURCE/'memory_preflight.py'),
 '--initialization','metric_outdoor','--metric-model',a['metric_model'],
 '--image-size','518','--batch-size','1','--max-depth','65',
 '--model-max-depth','80','--amp','--output',str(PREFLIGHT)],check=True)
assert json.loads(PREFLIGHT.read_text())['status']=='pass'


## 明示的に許可された停止epochまで再開
初期値5では学習を開始できない。総20epoch、1920予定stepsはそのまま。checkpoint、optimizer、scheduler、scaler、RNGは固定train.pyが復元する。

In [ ]:
STOP_AFTER_EPOCH=5  # Change only after the user explicitly authorizes the next stage.
assert 5<STOP_AFTER_EPOCH<=20, 'No additional epochs authorized in this template.'
args=dict(META['arguments'])
args.update(resume=True,stop_after_epoch=STOP_AFTER_EPOCH,memory_preflight_report=str(PREFLIGHT))
command=[sys.executable,'-u',str(SOURCE/'train.py')]
for key,value in args.items():
    flag='--'+key.replace('_','-')
    if isinstance(value,bool):
        if value: command.append(flag)
    elif value is not None: command += [flag,str(value)]
assert args['epochs']==20 and args['output_dir']==str(RUN)
subprocess.run(command,check=True)


終了時はcomplete.jsonと3ファイルのSHA256、run_status、Drive同期を確認する。GPUランタイムを終了してから結果をlocalへ取り込む。WindowsのC:\パスをColabへ渡さない。